In [1]:
# CELL 1 — INSTALL  (bỏ comment %%capture khi dùng Notebook)
import subprocess
import sys

def pip_install(cmd):
    subprocess.check_call([sys.executable, "-m", "pip"] + cmd)

print("Uninstall peft...")
pip_install(["uninstall", "-y", "peft"])

print("Installing dependencies...")

packages = [
    "transformers==4.37.2",
    "datasets==2.16.0",
    "accelerate==0.25.0",
    "peft==0.7.1",
    "evaluate",
    "scikit-learn",
    "pillow",
    "pytesseract"
]

for pkg in packages:
    print(f"Installing {pkg}...")
    pip_install(["install", pkg])

print("All packages installed successfully!")

Uninstall peft...
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
Installing dependencies...
Installing transformers==4.37.2...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Succe

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.


Installing datasets==2.16.0...
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
  Attempt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pathos 0.3.2 requires dill>=0.3.8, but you have dill 0.3.7 which is incompatible.
pathos 0.3.2 requires multiprocess>=0.70.16, but you have multiprocess 0.70.15 which is incompatible.
sentence-transformers 5.2.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2023.10.0 which is incompatible.


Installing accelerate==0.25.0...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 10.1 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
Installing peft==0.7.1...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 8.0 MB/s eta 0:00:00
Installing evaluate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
Installing scikit-learn...
Installing pillow...
Installing pytesseract...
All packages installed successfully!


In [2]:
# CELL 2 — CHECK GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

GPU: Tesla T4
Memory: 15.6GB


In [3]:
# ============================================================
# CELL 3 — IMPORTS & CONFIG
# ============================================================
import os, json, shutil, sys
import numpy as np
from pathlib import Path
from dataclasses import dataclass

from datasets import load_from_disk
from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from transformers.trainer_callback import EarlyStoppingCallback
from sklearn.metrics import precision_recall_fscore_support, accuracy_score


@dataclass
class Config:
    # ─────────────────────────────────────────────────────────
    # KAGGLE PATHS
    # Kiểm tra path đúng bằng: os.listdir("/kaggle/input/")
    # Kaggle tự lowercase + hyphen tên dataset
    # ─────────────────────────────────────────────────────────
    dataset_dir : str = "/kaggle/input/datasets/phanphat/hf-dataset-table/hf_dataset"
    model_name  : str = "/kaggle/input/datasets/phanphat/layoutlmv3/layoutlmv3-base"
    output_dir  : str = "/kaggle/working/layoutlmv3_invoice"

    # ── Hyperparameters ───────────────────────────────────────
    learning_rate               : float = 5e-5
    per_device_train_batch_size : int   = 4
    per_device_eval_batch_size  : int   = 8
    num_train_epochs            : int   = 15
    warmup_ratio                : float = 0.1
    weight_decay                : float = 0.01
    max_grad_norm               : float = 1.0

    # ── Early stopping ────────────────────────────────────────
    early_stopping_patience  : int   = 3
    early_stopping_threshold : float = 0.001

    # ── Eval & Save ───────────────────────────────────────────
    eval_strategy          : str  = "epoch"
    save_strategy          : str  = "epoch"
    save_total_limit       : int  = 2
    load_best_model_at_end : bool = True
    metric_for_best_model  : str  = "eval_f1"

    # ── Hardware ──────────────────────────────────────────────
    fp16                   : bool = True
    dataloader_num_workers : int  = 2
    logging_steps          : int  = 50


cfg = Config()

# ── VERIFY KAGGLE PATHS ────────────────────────────────────────
print("📁 /kaggle/input/ contents:")
try:
    for item in sorted(os.listdir("/kaggle/input/")):
        print(f"   {item}/")
except Exception as e:
    print(f"   (local env: {e})")

sys.stdout.flush()

2026-03-07 16:10:43.471900: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772899843.668500      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772899843.721329      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772899844.170675      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772899844.170729      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772899844.170732      24 computation_placer.cc:177] computation placer alr

📁 /kaggle/input/ contents:
   datasets/


In [4]:
# ============================================================
# CELL 4 — LOAD DATASET
#
# KEY POINT: prepare_layoutlmv3.py save từng split RIÊNG
#   hf_dataset/train/  → Dataset (không phải DatasetDict)
#   hf_dataset/val/    → Dataset
#   hf_dataset/test/   → Dataset
#
# → PHẢI load từng split riêng, KHÔNG load root
# ============================================================
print("\n" + "="*60)
print("STEP 1: LOAD DATASET")
print("="*60)

BASE = cfg.dataset_dir

# Load 3 splits riêng
train_ds = load_from_disk(f"{BASE}/train")
val_ds   = load_from_disk(f"{BASE}/val")
test_ds  = load_from_disk(f"{BASE}/test")

print(f"\n✅ Splits loaded:")
print(f"   train : {len(train_ds):>4} samples")
print(f"   val   : {len(val_ds):>4} samples")
print(f"   test  : {len(test_ds):>4} samples")

print(f"\n✅ Columns: {train_ds.column_names}")
# Expected: ['id', 'input_ids', 'attention_mask', 'bbox', 'labels', 'image']
# KHÔNG có 'words' hay 'bboxes'

# Verify 1 sample
s = train_ds[0]
print(f"\n✅ Sample check:")
print(f"   input_ids len : {len(s['input_ids'])}")
print(f"   bbox len      : {len(s['bbox'])}")
print(f"   labels len    : {len(s['labels'])}")
# Load label mappings
with open(f"{BASE}/label2id_table.json", encoding="utf-8") as f:
    label2id = json.load(f)
with open(f"{BASE}/id2label_table.json", encoding="utf-8") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}

num_labels = len(label2id)
print(f"\n✅ Labels: {num_labels}  (O = {label2id['O']})")
sys.stdout.flush()


STEP 1: LOAD DATASET

✅ Splits loaded:
   train : 1600 samples
   val   :  200 samples
   test  :  200 samples

✅ Columns: ['id', 'input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values']

✅ Sample check:
   input_ids len : 512
   bbox len      : 512
   labels len    : 512

✅ Labels: 25  (O = 24)


In [5]:
# ============================================================
# CELL 4.5 — COPY SANG KAGGLE/WORKING
# ============================================================
import os
import shutil
from datasets import load_from_disk

src = "/kaggle/input/datasets/phanphat/hf-dataset-table/hf_dataset"
dst = "/kaggle/working/hf_dataset"

if not os.path.exists(dst):
    print("Copying dataset to /working...")
    shutil.copytree(src, dst)
else:
    print("Dataset already exists in /working")

train_ds = load_from_disk(f"{dst}/train")
val_ds   = load_from_disk(f"{dst}/val")
test_ds  = load_from_disk(f"{dst}/test")

print("✅ Dataset ready from /working")


Copying dataset to /working...
✅ Dataset ready from /working


In [6]:
# ============================================================
# CELL 5 — LOAD PROCESSOR & MODEL
# ============================================================
print("\n" + "="*60)
print("STEP 2: LOAD MODEL & PROCESSOR")
print("="*60)

processor = LayoutLMv3Processor.from_pretrained(
    cfg.model_name,
    apply_ocr=False,
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    cfg.model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

print(f"✅ Processor loaded")
print(f"✅ Model: {model.num_parameters():,} parameters")
sys.stdout.flush()


STEP 2: LOAD MODEL & PROCESSOR


Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at /kaggle/input/datasets/phanphat/layoutlmv3/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Processor loaded
✅ Model: 125,936,793 parameters


In [7]:
# ============================================================
# CELL 6 — Custom DataCollator (NumPy 2.x safe)
# pixel_values đã có sẵn trong dataset từ prepare_layoutlmv3.py
# Không cần add_pixel_values nữa
# ============================================================
import torch
import numpy as np

REMOVE_COLS = ["id"]
train_ds = train_ds.remove_columns(REMOVE_COLS)
val_ds   = val_ds.remove_columns(REMOVE_COLS)
test_ds  = test_ds.remove_columns(REMOVE_COLS)

class LayoutLMv3DataCollator:
    """NumPy 2.x safe collator."""
    def __call__(self, features):
        return {
            "input_ids":      torch.tensor(
                                  [f["input_ids"] for f in features],
                                  dtype=torch.long),
            "attention_mask": torch.tensor(
                                  [f["attention_mask"] for f in features],
                                  dtype=torch.long),
            "bbox":           torch.tensor(
                                  [f["bbox"] for f in features],
                                  dtype=torch.long),
            "labels":         torch.tensor(
                                  [f["labels"] for f in features],
                                  dtype=torch.long),
            "pixel_values":   torch.tensor(
                                  np.array([f["pixel_values"] for f in features]),
                                  dtype=torch.float),
        }

data_collator = LayoutLMv3DataCollator()

# Verify
sample_batch = data_collator([train_ds[0], train_ds[1]])
print(f"✅ pixel_values : {sample_batch['pixel_values'].shape}")  # [2, 3, 224, 224]
print(f"✅ input_ids    : {sample_batch['input_ids'].shape}")     # [2, 512]
print(f"✅ DataCollator OK")

✅ pixel_values : torch.Size([2, 3, 224, 224])
✅ input_ids    : torch.Size([2, 512])
✅ DataCollator OK


In [8]:
# ============================================================
# CELL 7 — COLLATOR & METRICS
# ============================================================
# ✅ Dùng LayoutLMv3DataCollator từ Cell 6 (NumPy 2.x safe)
# ❌ default_data_collator → cùng lỗi copy=False với bbox ragged array
# ❌ DataCollatorForTokenClassification → DROP pixel_values

# data_collator đã được định nghĩa ở Cell 6
# Chỉ cần verify lại:
assert data_collator is not None, "Chạy Cell 6 trước!"

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)  # [batch, seq_len]

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        for p, l in zip(pred_seq, label_seq):
            if int(l) != -100:               # ✅ ép int — tránh numpy int64 key lookup
                true_preds.append(id2label[int(p)])
                true_labels.append(id2label[int(l)])

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, true_preds,
        average="weighted",
        zero_division=0,
    )
    accuracy = accuracy_score(true_labels, true_preds)

    return {
        "precision" : round(float(precision), 4),
        "recall"    : round(float(recall),    4),
        "f1"        : round(float(f1),        4),
        "accuracy"  : round(float(accuracy),  4),
    }

print("✅ data_collator  : LayoutLMv3DataCollator (NumPy 2.x safe)")
print("✅ compute_metrics: weighted F1, skip -100, int() safe key lookup")

✅ data_collator  : LayoutLMv3DataCollator (NumPy 2.x safe)
✅ compute_metrics: weighted F1, skip -100, int() safe key lookup


In [9]:
# ============================================================
# CELL 8 — TRAINING ARGUMENTS
# ============================================================
import transformers
from packaging import version

# ── Guards ───────────────────────────────────────────────────
assert "data_collator" in dir() and data_collator is not None, \
    "❌ Chạy Cell 6 trước — data_collator chưa định nghĩa!"

os.makedirs(cfg.output_dir, exist_ok=True)

# ✅ Kaggle: num_workers=0 tránh deadlock
SAFE_NUM_WORKERS = 0

# ✅ eval_strategy đổi tên từ transformers >= 4.41
# Phiên bản cũ hơn dùng evaluation_strategy
_transformers_version = version.parse(transformers.__version__)
_use_new_eval_arg     = _transformers_version >= version.parse("4.41.0")
_eval_strategy_key    = "eval_strategy" if _use_new_eval_arg else "evaluation_strategy"

print(f"   Transformers version : {transformers.__version__}")
print(f"   eval strategy key    : '{_eval_strategy_key}'")

# ✅ load_best_model_at_end YÊU CẦU eval_strategy == save_strategy
assert cfg.eval_strategy == cfg.save_strategy, (
    f"❌ eval_strategy='{cfg.eval_strategy}' ≠ save_strategy='{cfg.save_strategy}'\n"
    f"   load_best_model_at_end=True yêu cầu hai strategy phải giống nhau!"
)

training_args = TrainingArguments(
    output_dir=cfg.output_dir,

    # ── Training ─────────────────────────────────────────────
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    learning_rate=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    max_grad_norm=cfg.max_grad_norm,

    # ── Eval & Save (version-safe) ───────────────────────────
    **{_eval_strategy_key: cfg.eval_strategy},  # ✅ tự chọn đúng key theo version
    save_strategy=cfg.save_strategy,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=cfg.load_best_model_at_end,
    metric_for_best_model=cfg.metric_for_best_model,
    greater_is_better=True,

    # ── Logging ──────────────────────────────────────────────
    logging_steps=cfg.logging_steps,
    logging_dir=f"{cfg.output_dir}/logs",
    report_to="none",

    # ── Hardware ─────────────────────────────────────────────
    fp16=cfg.fp16,
    dataloader_num_workers=SAFE_NUM_WORKERS,
    seed=42,

    # ── Label alignment ──────────────────────────────────────
    label_names=["labels"],
)

# ── Summary ──────────────────────────────────────────────────
n_steps     = len(train_ds) // cfg.per_device_train_batch_size
total_steps = n_steps * cfg.num_train_epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)

print("\n📋 Training config:")
print(f"   Epochs            : {cfg.num_train_epochs}")
print(f"   Batch size        : {cfg.per_device_train_batch_size}")
print(f"   Steps/epoch       : {n_steps}")
print(f"   Total steps       : {total_steps}")
print(f"   Warmup steps      : {warmup_steps}  ({cfg.warmup_ratio:.0%})")
print(f"   LR                : {cfg.learning_rate}")
print(f"   fp16              : {cfg.fp16}")
print(f"   {_eval_strategy_key:<17} : {cfg.eval_strategy}")
print(f"   save_strategy     : {cfg.save_strategy}  ✅ match")
print(f"   Early stop        : patience={cfg.early_stopping_patience}")
print(f"   save_total_limit  : {cfg.save_total_limit}")
print(f"   dataloader_workers: {SAFE_NUM_WORKERS}  (Kaggle safe)")
print(f"   label_names       : ['labels']")
print(f"\n⚠️  REMINDER — Cell 9 Trainer phải có:")
print(f"   data_collator=data_collator   ← LayoutLMv3DataCollator từ Cell 6")
sys.stdout.flush()

   Transformers version : 4.37.2
   eval strategy key    : 'evaluation_strategy'

📋 Training config:
   Epochs            : 15
   Batch size        : 4
   Steps/epoch       : 400
   Total steps       : 6000
   Warmup steps      : 600  (10%)
   LR                : 5e-05
   fp16              : True
   evaluation_strategy : epoch
   save_strategy     : epoch  ✅ match
   Early stop        : patience=3
   save_total_limit  : 2
   dataloader_workers: 0  (Kaggle safe)
   label_names       : ['labels']

⚠️  REMINDER — Cell 9 Trainer phải có:
   data_collator=data_collator   ← LayoutLMv3DataCollator từ Cell 6


In [10]:
# ============================================================
# CELL 9 — TRAIN  (fixed for transformers==4.37.2)
# ============================================================
import transformers
from packaging import version

# ── Guards ───────────────────────────────────────────────────
assert "data_collator"   in dir() and data_collator   is not None, \
    "❌ Chạy Cell 6 — data_collator chưa định nghĩa!"
assert "compute_metrics" in dir() and compute_metrics is not None, \
    "❌ Chạy Cell 7 — compute_metrics chưa định nghĩa!"
assert "training_args"   in dir() and training_args   is not None, \
    "❌ Chạy Cell 8 — training_args chưa định nghĩa!"
assert "model"           in dir() and model           is not None, \
    "❌ Model chưa load!"

# ── Version-safe: processing_class chỉ có từ transformers >= 4.45 ──
_transformers_version  = version.parse(transformers.__version__)
_use_processing_class  = _transformers_version >= version.parse("4.45.0")

print(f"   Transformers version : {transformers.__version__}")
print(f"   Trainer arg          : {'processing_class' if _use_processing_class else 'tokenizer'}")

print("\n" + "="*60)
print("STEP 4: TRAINING")
print("="*60)
print(f"  train   : {len(train_ds)} samples")
print(f"  val     : {len(val_ds)} samples")
print(f"  collator: {type(data_collator).__name__}")
sys.stdout.flush()

# ── Build trainer kwargs version-safe ────────────────────────
_trainer_processor_kwargs = (
    {"processing_class": processor}           # transformers >= 4.45
    if _use_processing_class else
    {"tokenizer": processor.tokenizer}        # transformers == 4.37.2 ✅
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    **_trainer_processor_kwargs,              # ✅ tự chọn đúng theo version
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=cfg.early_stopping_patience,
            early_stopping_threshold=cfg.early_stopping_threshold,
        )
    ],
)

# ── Train ────────────────────────────────────────────────────
train_result = trainer.train()

# ── Save ngay (quan trọng trên Kaggle) ───────────────────────
trainer.save_model(cfg.output_dir)
trainer.save_state()

# ── Log history ──────────────────────────────────────────────
import json as _json
log_path = f"{cfg.output_dir}/train_log_history.json"
with open(log_path, "w") as f:
    _json.dump(trainer.state.log_history, f, indent=2)

# ── Summary ──────────────────────────────────────────────────
metrics = train_result.metrics
print("\n✅ Training done!")
print(f"   train_loss        : {metrics.get('train_loss', 0):.4f}")
print(f"   train_runtime     : {metrics.get('train_runtime', 0)/3600:.2f}h")
print(f"   samples/sec       : {metrics.get('train_samples_per_second', 0):.1f}")
print(f"   steps/sec         : {metrics.get('train_steps_per_second', 0):.2f}")
print(f"   Best model saved  : {cfg.output_dir}")
print(f"   Log history saved : {log_path}")
sys.stdout.flush()

   Transformers version : 4.37.2
   Trainer arg          : tokenizer

STEP 4: TRAINING
  train   : 1600 samples
  val     : 200 samples
  collator: LayoutLMv3DataCollator


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.063500,0.057594,0.981200,0.984100,0.980300,0.984100
2,0.010300,0.007800,0.998300,0.998800,0.998500,0.998800
3,0.011400,0.004125,0.999000,0.999300,0.999100,0.999300
4,0.006900,0.003807,0.999400,0.999400,0.999400,0.999400
5,0.005000,0.002523,0.999500,0.999500,0.999500,0.999500


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warn


✅ Training done!
   train_loss        : 0.1902
   train_runtime     : 0.47h
   samples/sec       : 14.2
   steps/sec         : 1.77
   Best model saved  : /kaggle/working/layoutlmv3_invoice
   Log history saved : /kaggle/working/layoutlmv3_invoice/train_log_history.json


In [11]:
# ============================================================
# CELL 10 — EVALUATE
# ============================================================

# ── Guards ───────────────────────────────────────────────────
assert "trainer"      in dir() and trainer      is not None, \
    "❌ Chạy Cell 9 — trainer chưa định nghĩa!"
assert "test_ds"      in dir() and test_ds      is not None, \
    "❌ test_ds chưa load!"
assert "data_collator" in dir() and data_collator is not None, \
    "❌ Chạy Cell 6 — data_collator chưa định nghĩa!"

import json as _json
from sklearn.metrics import classification_report

print("\n" + "="*60)
print("STEP 5: TEST EVALUATION")
print("="*60)
print(f"  test : {len(test_ds)} samples")
sys.stdout.flush()

# ── Evaluate ─────────────────────────────────────────────────
test_results = trainer.evaluate(test_ds)

# ── Save kết quả ngay (Kaggle session có thể expire) ─────────
results_path = f"{cfg.output_dir}/test_results.json"
with open(results_path, "w") as f:
    _json.dump(test_results, f, indent=2)

# ── Summary metrics ──────────────────────────────────────────
print("\n📊 Test Results (weighted):")
print(f"   F1        : {test_results.get('eval_f1',        0):.4f}")
print(f"   Precision : {test_results.get('eval_precision', 0):.4f}")
print(f"   Recall    : {test_results.get('eval_recall',    0):.4f}")
print(f"   Accuracy  : {test_results.get('eval_accuracy',  0):.4f}")
print(f"   Loss      : {test_results.get('eval_loss',      0):.4f}")
print(f"   Runtime   : {test_results.get('eval_runtime',   0):.1f}s")
print(f"\n   Saved → {results_path}")

# ── Per-class F1 report (debug field nào yếu) ────────────────
print("\n📋 Per-class Report:")
print("   (chạy inference thủ công để lấy token-level predictions)")

# Dùng lại logic compute_metrics nhưng collect all preds/labels
all_preds, all_labels = [], []

# DataLoader thủ công với data_collator từ Cell 6
from torch.utils.data import DataLoader

test_loader = DataLoader(
    test_ds,
    batch_size=cfg.per_device_eval_batch_size,
    collate_fn=data_collator,    # ✅ LayoutLMv3DataCollator — NumPy 2.x safe
    shuffle=False,
    num_workers=0,               # ✅ Kaggle safe
)

model.eval()
device = next(model.parameters()).device

with torch.no_grad():
    for batch in test_loader:
        # Move to device
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        logits  = outputs.logits                          # [B, seq, num_labels]
        labels  = batch["labels"]                         # [B, seq]
        preds   = torch.argmax(logits, dim=-1)            # [B, seq]

        for pred_seq, label_seq in zip(preds, labels):
            for p, l in zip(pred_seq, label_seq):
                l_int = int(l)
                if l_int != -100:                         # ✅ skip padding
                    all_preds.append(id2label[int(p)])
                    all_labels.append(id2label[l_int])

# ── Per-class classification report ─────────────────────────
# Lấy label list đúng thứ tự (bỏ O để dễ đọc, hiện ở cuối)
target_names_ordered = (
    [v for v in id2label.values() if v != "O"] + ["O"]
)

report = classification_report(
    all_labels,
    all_preds,
    labels=target_names_ordered,
    zero_division=0,
    digits=4,
)
print(report)

# ── Save per-class report ─────────────────────────────────────
report_path = f"{cfg.output_dir}/test_per_class_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("TEST EVALUATION — PER CLASS REPORT\n")
    f.write("="*60 + "\n")
    f.write(f"F1 weighted : {test_results.get('eval_f1', 0):.4f}\n")
    f.write(f"Accuracy    : {test_results.get('eval_accuracy', 0):.4f}\n")
    f.write("="*60 + "\n\n")
    f.write(report)

print(f"\n✅ Per-class report saved → {report_path}")
sys.stdout.flush()


STEP 5: TEST EVALUATION
  test : 200 samples


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(



📊 Test Results (weighted):
   F1        : 0.9996
   Precision : 0.9996
   Recall    : 0.9996
   Accuracy  : 0.9996
   Loss      : 0.0043
   Runtime   : 12.4s

   Saved → /kaggle/working/layoutlmv3_invoice/test_results.json

📋 Per-class Report:
   (chạy inference thủ công để lấy token-level predictions)


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:993: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


                    precision    recall  f1-score   support

       B-ITEM_NAME     0.9986    1.0000    0.9993      1467
       B-ITEM_UNIT     1.0000    1.0000    1.0000      1285
   B-ITEM_QUANTITY     1.0000    1.0000    1.0000      1206
 B-ITEM_UNIT_PRICE     1.0000    0.9992    0.9996      1274
B-ITEM_TOTAL_PRICE     0.9961    1.0000    0.9980      1261
   B-ITEM_DISCOUNT     0.9928    1.0000    0.9964       138
   B-ITEM_VAT_RATE     1.0000    0.9893    0.9946       374
   B-ITEM_LINE_TAX     1.0000    1.0000    1.0000       376
  B-ITEM_ROW_TOTAL     1.0000    1.0000    1.0000       135
        B-SUBTOTAL     1.0000    1.0000    1.0000        92
      B-VAT_AMOUNT     1.0000    1.0000    1.0000        91
     B-GRAND_TOTAL     1.0000    1.0000    1.0000       131
       I-ITEM_NAME     1.0000    0.9997    0.9998      6010
       I-ITEM_UNIT     1.0000    1.0000    1.0000         4
   I-ITEM_QUANTITY     0.0000    0.0000    0.0000         0
 I-ITEM_UNIT_PRICE     1.0000    1.0000

In [12]:
# ============================================================
# CELL 11 — SAVE & ZIP
# ============================================================

# ── Guards ───────────────────────────────────────────────────
assert "trainer"      in dir() and trainer      is not None, \
    "❌ Chạy Cell 9 — trainer chưa định nghĩa!"
assert "train_result" in dir() and train_result is not None, \
    "❌ Chạy Cell 9 — train_result chưa định nghĩa!"
assert "test_results" in dir() and test_results is not None, \
    "❌ Chạy Cell 10 — test_results chưa định nghĩa!"
assert "processor"    in dir() and processor    is not None, \
    "❌ processor chưa load!"

import shutil
import json as _json
from pathlib import Path

print("\n" + "="*60)
print("STEP 6: SAVE MODEL")
print("="*60)

final_dir = Path(f"{cfg.output_dir}/final_model")
final_dir.mkdir(parents=True, exist_ok=True)

# ── 1. Save model + processor ────────────────────────────────
trainer.save_model(str(final_dir))          # config.json, pytorch_model.bin
processor.save_pretrained(str(final_dir))   # tokenizer, image_processor

# ── 2. Save label maps ───────────────────────────────────────
with open(final_dir / "label2id.json", "w", encoding="utf-8") as f:
    _json.dump(label2id, f, indent=2, ensure_ascii=False)
with open(final_dir / "id2label.json", "w", encoding="utf-8") as f:
    _json.dump(id2label, f, indent=2, ensure_ascii=False)

# ── 3. Save metrics (train + test) ───────────────────────────
with open(final_dir / "metrics.json", "w", encoding="utf-8") as f:
    _json.dump(
        {
            "train": train_result.metrics,
            "test":  test_results,
        },
        f, indent=2, ensure_ascii=False,
    )

# ── 4. Copy log/report từ Cell 9 & 10 vào final_dir ─────────
#    Đảm bảo zip đầy đủ khi download
_extra_files = [
    Path(cfg.output_dir) / "train_log_history.json",    # Cell 9
    Path(cfg.output_dir) / "test_results.json",          # Cell 10
    Path(cfg.output_dir) / "test_per_class_report.txt",  # Cell 10
]
for src in _extra_files:
    if src.exists():
        shutil.copy2(src, final_dir / src.name)
        print(f"   📋 Copied: {src.name}")
    else:
        print(f"   ⚠️  Missing (skip): {src.name}")

# ── 5. List saved files ──────────────────────────────────────
print("\n✅ Saved files:")
total_size = 0
for fp in sorted(final_dir.iterdir()):
    if fp.is_file():
        size_mb = fp.stat().st_size / 1e6
        total_size += size_mb
        print(f"   {fp.name:<42} {size_mb:>7.1f} MB")
print(f"   {'─'*50}")
print(f"   {'TOTAL':<42} {total_size:>7.1f} MB")

# ── 6. Zip final_dir → /kaggle/working/ ──────────────────────
zip_path   = Path("/kaggle/working/model_final")
zip_source = final_dir
zip_size_mb = 0

try:
    shutil.make_archive(
        base_name=str(zip_path),   # output: /kaggle/working/model_final.zip
        format="zip",
        root_dir=str(final_dir.parent),  # ✅ zip relative path
        base_dir=final_dir.name,         # ✅ zip thành folder "final_model/"
    )
    zip_size_mb = (zip_path.with_suffix(".zip")).stat().st_size / 1e6
    print(f"\n✅ model_final.zip ({zip_size_mb:.1f} MB)")
    print("📥 Download: Output tab → model_final.zip")

except Exception as e:
    print(f"\n⚠️  Zip failed: {e}")
    print("   Manual zip fallback:")
    print(f"   !zip -r /kaggle/working/model_final.zip {final_dir}")

# ── 7. Final summary ─────────────────────────────────────────
print("\n" + "="*60)
print("🎉 ALL DONE")
print("="*60)
print(f"   F1        : {test_results.get('eval_f1',        0)*100:.2f}%")
print(f"   Precision : {test_results.get('eval_precision', 0)*100:.2f}%")
print(f"   Recall    : {test_results.get('eval_recall',    0)*100:.2f}%")
print(f"   Accuracy  : {test_results.get('eval_accuracy',  0)*100:.2f}%")
print(f"   Loss      : {test_results.get('eval_loss',      0):.4f}")
print(f"   Model dir : {final_dir}")
print(f"   Zip size  : {zip_size_mb:.1f} MB")
print("="*60)
sys.stdout.flush()


STEP 6: SAVE MODEL
   📋 Copied: train_log_history.json
   📋 Copied: test_results.json
   📋 Copied: test_per_class_report.txt

✅ Saved files:
   config.json                                    0.0 MB
   id2label.json                                  0.0 MB
   label2id.json                                  0.0 MB
   merges.txt                                     0.5 MB
   metrics.json                                   0.0 MB
   model.safetensors                            503.8 MB
   preprocessor_config.json                       0.0 MB
   special_tokens_map.json                        0.0 MB
   test_per_class_report.txt                      0.0 MB
   test_results.json                              0.0 MB
   tokenizer.json                                 2.1 MB
   tokenizer_config.json                          0.0 MB
   train_log_history.json                         0.0 MB
   training_args.bin                              0.0 MB
   vocab.json                                     0.8 MB
   